# Lab 9: Der ML-Workflow als Pipeline

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Datenordner finden: Notebook liegt in labs/ oder loesungen/, die Daten in data/
DATA = next(p for p in [Path("data"), Path("../data"), Path("../../data")] if p.exists())
print("Datenordner:", DATA)

Dieses Lab gehört zu **Teil 9: Der ML-Workflow mit scikit-learn**.

## Lernziele

- Sie erzeugen ein Datenleck selbst und sehen, welche Größen dabei aus den Testdaten ins Training gelangen.
- Sie bauen eine `Pipeline` aus Aufbereitung und Modell und greifen auf einzelne Schritte zu.
- Sie behandeln Zahlen- und Kategoriespalten mit `ColumnTransformer` getrennt.
- Sie bewerten die ganze Pipeline mit `cross_val_score` und stellen sie mit `GridSearchCV` ein.
- Sie speichern die fertige Pipeline mit `joblib`, laden sie wieder und sagen neue Passagiere vorher.

Arbeitsweise: Jeder Block beginnt mit einer Aufgabenliste. Unter jeder Aufgabe steht ein Kontrollergebnis, mit dem Sie sich selbst prüfen. Die Zahlen gelten für `random_state=1`.

In [ ]:
import tempfile
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

df = pd.read_csv(DATA / "titanic.csv")
print(df.shape)
df.head()

## Block 1: Das Datenleck selbst erzeugen

Sie arbeiten zuerst nur mit vier Zahlenspalten. `Age` hat fehlende Werte, deshalb brauchen Sie einen Imputer und danach einen Scaler. Die Frage ist, **wann** beide ihre Kennwerte lernen: vor oder nach dem Split.

1. **Mit Leck:** Füllen und skalieren Sie `X1` auf allen Zeilen, teilen Sie erst danach (`test_size=0.2`, `random_state=1`, `stratify=y`) und trainieren Sie eine logistische Regression. Erwartet: Accuracy auf den Testdaten 0.693.
2. **Ohne Leck:** Teilen Sie zuerst. Imputer und Scaler lernen mit `fit_transform` nur auf den Trainingsdaten, die Testdaten bekommen nur `transform`. Erwartet: Accuracy 0.693.
3. Vergleichen Sie, was Imputer und Scaler in beiden Fassungen gelernt haben: den Median von `Age` und die Mittelwerte des Scalers. Erwartet: Median von `Age` 28.0 mit Leck und 29.0 ohne Leck, Mittelwert von `Fare` 32.20 mit Leck und 31.93 ohne Leck.

Beobachtung: Die Accuracy ist hier in beiden Fassungen gleich, die gelernten Kennwerte sind es nicht. Beim Skalieren bleibt der Schaden klein. In der Zusatzaufgabe 3 sehen Sie einen Fall, in dem dasselbe Muster die Kennzahl völlig verfälscht.

In [ ]:
zahlen_spalten = ["Age", "Fare", "SibSp", "Parch"]
X1 = df[zahlen_spalten]
y = df["Survived"]
print(X1.isna().sum())

In [ ]:
# Aufgabe 1: Fassung MIT Leck
# Schritte: SimpleImputer(strategy="median") und StandardScaler mit fit_transform auf ALLEN Zeilen,
#           danach train_test_split, danach LogisticRegression(max_iter=1000)
# Tipp: imputer_leck.fit_transform(X1), scaler_leck.fit_transform(...), modell.score(...)
imputer_leck = ...
scaler_leck = ...
acc_leck = ...
print("Accuracy mit Leck:", acc_leck)

In [ ]:
# Aufgabe 2: Fassung OHNE Leck
# Schritte: zuerst train_test_split auf X1 und y, dann fit_transform nur auf X1_train,
#           für X1_test nur transform
# X1_train, X1_test, y1_train, y1_test = train_test_split(...)
imputer_ok = ...
scaler_ok = ...
acc_ok = ...
print("Accuracy ohne Leck:", acc_ok)

In [ ]:
# Aufgabe 3: Was haben Imputer und Scaler gelernt?
# Tipp: imputer.statistics_ enthält die Mediane, scaler.mean_ die Mittelwerte
# Bauen Sie eine Tabelle mit den Spalten "mit Leck" und "ohne Leck"
vergleich = ...
vergleich

## Block 2: Erste Pipeline mit Zahlenspalten

Die saubere Fassung aus Block 1 braucht für jeden Schritt zwei Aufrufe und zwei Varianten jeder Matrix. Eine `Pipeline` nimmt Ihnen diese Buchführung ab. Sie verwenden weiter `X1_train`, `X1_test`, `y1_train`, `y1_test` aus Block 1.

1. Bauen Sie eine Pipeline aus `StandardScaler` (Name `"scaler"`) und `LogisticRegression(max_iter=1000)` (Name `"modell"`). Trainieren Sie nur mit den Spalten ohne Lücken: `Fare`, `SibSp`, `Parch`. Erwartet: Accuracy auf den Testdaten 0.682.
2. Lesen Sie über `named_steps` die gelernten Mittelwerte des Scalers aus. Stellen Sie danach mit `set_params(modell__C=0.01)` den Parameter `C` um, trainieren Sie neu und vergleichen Sie. Erwartet: Mittelwert von `Fare` 31.93, Accuracy mit `C=0.01` 0.659.
3. Nehmen Sie `Age` dazu. Die Pipeline aus Aufgabe 1 bricht jetzt ab, weil `Age` fehlende Werte hat. Fangen Sie den Fehler mit `try/except` ab und ergänzen Sie danach einen `SimpleImputer(strategy="median")` als ersten Schritt. Erwartet: Accuracy 0.693, dieselbe Zahl wie in Block 1, diesmal mit drei Zeilen Code.

In [ ]:
# Aufgabe 1: Pipeline aus Scaler und logistischer Regression
ohne_luecken = ["Fare", "SibSp", "Parch"]
# Tipp: Pipeline([("scaler", ...), ("modell", ...)]), danach fit und score mit den ROHEN Daten
pipe = ...
# pipe.fit(X1_train[ohne_luecken], y1_train)
# print(round(pipe.score(X1_test[ohne_luecken], y1_test), 3))

In [ ]:
# Aufgabe 2: Auf Schritte zugreifen und einen Parameter ändern
# Tipp: pipe.named_steps["scaler"].mean_ und pipe.set_params(modell__C=0.01)
mittelwerte = ...
print(mittelwerte)
# Danach: Parameter setzen, neu trainieren, score ausgeben

In [ ]:
# Aufgabe 3: Age dazunehmen
# Teil a: pipe.fit(X1_train, y1_train) in try/except ValueError ausführen und die Meldung ausgeben
# Teil b: neue Pipeline pipe_imp mit den Schritten "imputer", "scaler", "modell"
pipe_imp = ...
# pipe_imp.fit(X1_train, y1_train)
# print(round(pipe_imp.score(X1_test, y1_test), 3))

## Block 3: ColumnTransformer für Zahlen und Kategorien

Mit Zahlenspalten allein bleibt das Modell schwach. Geschlecht, Klasse und Hafen sind Kategorien und brauchen eine andere Aufbereitung als Zahlen. Ab hier bauen Sie das Skript auf, das bis zum Ende des Labs weiterläuft.

1. Legen Sie `num_cols = ["Age", "Fare", "SibSp", "Parch"]` und `cat_cols = ["Pclass", "Sex", "Embarked"]` fest, bilden Sie `X` und `y` und teilen Sie mit `test_size=0.2`, `random_state=1`, `stratify=y`. Erwartet: `(712, 7)` und `(179, 7)`, Anteil Überlebender 0.383 im Training und 0.385 im Test.
2. Bauen Sie `numeric_pipe` (Imputer mit Median, dann Scaler), `categorical_pipe` (Imputer mit `most_frequent`, dann `OneHotEncoder(handle_unknown="ignore")`) und daraus `preprocess`. Sehen Sie sich mit `fit_transform(X_train)` Form und Spaltennamen an. Erwartet: `(712, 12)`, erste Spalte `num__Age`, letzte Spalte `cat__Embarked_S`.
3. Setzen Sie `preprocess` und `LogisticRegression(max_iter=1000)` zur Pipeline `clf` zusammen und trainieren Sie. Geben Sie die Accuracy auf den **Trainingsdaten** aus. Erwartet: 0.806. Die Testdaten bleiben ab jetzt bis Block 5 unberührt.

In [ ]:
# Aufgabe 1: Spalten wählen und teilen
num_cols = ["Age", "Fare", "SibSp", "Parch"]
cat_cols = ["Pclass", "Sex", "Embarked"]
X = ...
y = ...
# X_train, X_test, y_train, y_test = train_test_split(...)
# print(X_train.shape, X_test.shape)
# print(round(y_train.mean(), 3), round(y_test.mean(), 3))

In [ ]:
# Aufgabe 2: zwei kleine Pipelines und der ColumnTransformer
# Tipp: ColumnTransformer([("num", numeric_pipe, num_cols), ("cat", categorical_pipe, cat_cols)])
numeric_pipe = ...
categorical_pipe = ...
preprocess = ...
# Xt = preprocess.fit_transform(X_train)
# print(Xt.shape)
# print(preprocess.get_feature_names_out())

In [ ]:
# Aufgabe 3: vollständige Pipeline mit logistischer Regression
# Tipp: Schrittnamen "prep" und "modell"
clf = ...
# clf.fit(X_train, y_train)
# print("Accuracy Training:", round(clf.score(X_train, y_train), 3))
clf

## Block 4: Modell tauschen und mit Cross-Validation bewerten

Während der Entwicklung bewerten Sie nur über Cross-Validation auf den Trainingsdaten. In jedem der fünf Durchgänge lernt die Pipeline Median, Skalierung und Kategorien neu, nur aus dem jeweiligen Trainingsteil.

1. Bewerten Sie `clf` mit `cross_val_score` (`cv=5`, `scoring="accuracy"`) und geben Sie Mittelwert und Streuung aus. Erwartet: 0.795 +/- 0.018.
2. Bauen Sie die Pipeline `rf`: gleiche Aufbereitung, als Modell `RandomForestClassifier(n_estimators=200, random_state=1)`. Bewerten Sie genauso. Erwartet: 0.780 +/- 0.031.
3. Wählen Sie selbst ein drittes Modell, zum Beispiel `KNeighborsClassifier(n_neighbors=7)`, und stellen Sie alle drei Ergebnisse in einer Tabelle zusammen. Erwartet für k-NN mit 7 Nachbarn: 0.784 +/- 0.023.

In [ ]:
# Aufgabe 1: Cross-Validation der ganzen Pipeline
# Tipp: cross_val_score(clf, X_train, y_train, cv=5, scoring="accuracy")
scores_clf = ...
# print(f"LogReg: {scores_clf.mean():.3f} +/- {scores_clf.std():.3f}")

In [ ]:
# Aufgabe 2: Random Forest als letzter Schritt
rf = ...
scores_rf = ...
# print(f"RandomForest: {scores_rf.mean():.3f} +/- {scores_rf.std():.3f}")

In [ ]:
# Aufgabe 3: drittes Modell und Vergleichstabelle
# Tipp: eine Schleife über ein Dictionary {"Name": pipeline}, Ergebnisse in einer Liste sammeln
tabelle = ...
tabelle

## Block 5: GridSearchCV über Pipeline-Parameter

Der Random Forest mit Standardeinstellungen liegt hinter der logistischen Regression und schwankt stärker. Sie stellen ihn jetzt ein. Parameter eines Schritts sprechen Sie mit `schrittname__parameter` an.

1. Suchen Sie mit `GridSearchCV` über `modell__max_depth` aus `[3, 5, 8, None]` und `modell__n_estimators` aus `[100, 200]` (`cv=5`, `scoring="accuracy"`, `n_jobs=-1`). Erwartet: beste Einstellung `max_depth=5`, `n_estimators=200`, bester Mittelwert 0.81. Das sind 8 Kombinationen mal 5 Durchgänge, also 40 Trainingsläufe.
2. Lesen Sie `cv_results_` als Tabelle, sortiert nach `rank_test_score`. Erwartet: Die ersten Plätze liegen enger beieinander als ihre Streuung (`std_test_score` um 0.03).
3. Werten Sie `search.best_estimator_` **einmal** auf dem Testset aus: Konfusionsmatrix und `classification_report`. Erwartet: Matrix `[[101 9] [19 50]]`, Accuracy 0.84, Recall der Klasse 1 (überlebt) 0.72.

In [ ]:
# Aufgabe 1: kleines Gitter über zwei Parameter des Schritts "modell"
param_grid = {
    # "modell__max_depth": [...],
    # "modell__n_estimators": [...],
}
search = ...
# search.fit(X_train, y_train)
# print(search.best_params_)
# print(round(search.best_score_, 3))

In [ ]:
# Aufgabe 2: Ergebnisse der Suche als Tabelle
# Tipp: pd.DataFrame(search.cv_results_), Spalten param_..., mean_test_score, std_test_score, rank_test_score
ergebnis = ...
ergebnis

In [ ]:
# Aufgabe 3: bestes Modell einmal auf dem Testset auswerten
best = ...
# y_pred = best.predict(X_test)
# print(confusion_matrix(y_test, y_pred))
# print(classification_report(y_test, y_pred, digits=2))

## Block 6: Speichern, laden, vorhersagen

Eine `.joblib`-Datei enthält Imputer, Scaler, Encoder und Modell mit allen gelernten Größen. Die Datei schreiben Sie in ein temporäres Verzeichnis, das am Ende des Labs wieder gelöscht wird. Laden Sie `.joblib`-Dateien nur aus vertrauenswürdiger Quelle.

1. Speichern Sie `best` mit `joblib.dump` unter `pfad`, laden Sie die Datei als `geladen` und prüfen Sie die Accuracy auf dem Testset. Erwartet: 0.844, dieselbe Zahl wie vor dem Speichern.
2. Legen Sie einen neuen Passagier als DataFrame mit **einer Zeile** an (30 Jahre, Ticketpreis 50, allein reisend, erste Klasse, weiblich, Hafen `S`) und sagen Sie mit `predict` und `predict_proba` vorher. Erwartet: Klasse 1, Wahrscheinlichkeit für Überleben 0.94. Ändern Sie danach `Sex` auf `"male"` und `Pclass` auf 3. Erwartet: Klasse 0, Wahrscheinlichkeit für Überleben 0.21.
3. Sagen Sie eine Passagierin der zweiten Klasse (Ticketpreis 20, ein Geschwister oder Ehepartner an Bord) mit fehlendem Alter (`np.nan`) und unbekanntem Hafen `"X"` vorher. Begründen Sie in einem Kommentar, warum kein Fehler kommt. Erwartet: keine Fehlermeldung, Klasse 1 mit Wahrscheinlichkeit 0.88, die drei Hafen-Spalten der aufbereiteten Zeile sind alle 0.

In [ ]:
tmpdir = Path(tempfile.mkdtemp())
pfad = tmpdir / "titanic_pipeline.joblib"
print(pfad.name)

In [ ]:
# Aufgabe 1: speichern, laden, prüfen
# Tipp: joblib.dump(objekt, pfad) und joblib.load(pfad)
geladen = ...
# print(round(geladen.score(X_test, y_test), 3))

In [ ]:
# Aufgabe 2: neuer Passagier als DataFrame mit einer Zeile
# Tipp: pd.DataFrame([{...}]) mit denselben Spaltennamen wie im Training
neu = ...
# print(geladen.predict(neu))
# print(geladen.predict_proba(neu).round(2))

In [ ]:
# Aufgabe 3: fehlendes Alter und unbekannter Hafen
unbekannt = ...
# print(geladen.predict(unbekannt), geladen.predict_proba(unbekannt).round(2))
# Aufbereitete Zeile ansehen: geladen[:-1].transform(unbekannt)
# Begründung:

## Zusatzaufgaben

1. Nehmen Sie die Strategie des Imputers in die Suche auf: `prep__num__imputer__strategy` mit `"mean"` und `"median"`, dazu `modell__max_depth` aus `[3, 5, 8]`. Erwartet: bester Mittelwert 0.81 bei `max_depth=5`. `"mean"` und `"median"` liegen dort nur 0.001 auseinander (0.809 und 0.810), die Wahl der Strategie spielt hier kaum eine Rolle.
2. Gewinnen Sie aus `Name` den Titel (`Mr`, `Mrs`, `Miss`, `Master`, alle übrigen als `Sonstige`) als zusätzliche Kategoriespalte und prüfen Sie per Cross-Validation, ob die logistische Regression besser wird. Erwartet: 0.820 statt 0.795.
3. Ein Leck mit großer Wirkung: Erzeugen Sie 100 Zeilen mit 5000 Spalten reiner Zufallszahlen und eine zufällige Zielgröße. Wählen Sie mit `SelectKBest(f_classif, k=20)` die 20 „besten" Spalten einmal auf allen Zeilen aus (Leck) und einmal als Schritt einer Pipeline. Vergleichen Sie die Accuracy per Cross-Validation. Erwartet: mit Leck 0.89, in der Pipeline 0.52, obwohl in den Daten nichts zu lernen ist.

In [ ]:
# Zusatz 1: Imputer-Strategie in die Suche aufnehmen
param_grid_2 = {
    # "prep__num__imputer__strategy": [...],
    # "modell__max_depth": [...],
}
search_2 = ...

In [ ]:
# Zusatz 2: Titel aus dem Namen als neue Kategoriespalte
# Tipp: df["Name"].str.extract(r",\s*([^.]+)\.")[0] liefert den Text zwischen Komma und Punkt
titel = ...

In [ ]:
# Zusatz 3: Merkmalsauswahl auf Zufallszahlen, einmal mit Leck, einmal in der Pipeline
from sklearn.feature_selection import SelectKBest, f_classif

rng = np.random.default_rng(42)
X_rausch = rng.normal(size=(100, 5000))
y_rausch = rng.integers(0, 2, size=100)
# Mit Leck: SelectKBest(f_classif, k=20).fit_transform(X_rausch, y_rausch), danach cross_val_score
# Ohne Leck: Pipeline([("auswahl", SelectKBest(...)), ("modell", LogisticRegression(max_iter=1000))])

In [ ]:
# Aufräumen: temporäres Verzeichnis mit der gespeicherten Pipeline löschen
shutil.rmtree(tmpdir, ignore_errors=True)

## Was Sie mitnehmen

- `fit` und `fit_transform` sehen nur Trainingsdaten. Steht die ganze Aufbereitung in der Pipeline, gilt das von selbst, auch in jedem Durchgang von `cross_val_score` und `GridSearchCV`.
- `ColumnTransformer` gibt Zahlen und Kategorien getrennte Wege. `handle_unknown="ignore"` und der Imputer sorgen dafür, dass neue Daten mit Lücken oder unbekannten Kategorien keine Fehlermeldung auslösen.
- Verglichen und eingestellt wird über Cross-Validation auf den Trainingsdaten. Das Testset kommt genau einmal am Ende, danach wird die Pipeline als eine Datei gespeichert.